# Pipeline Agentique de Revue Systématique de Littérature (PRISMA 2020)

## 1. Installation des dépendances

In [1]:
!pip install -q langgraph langchain langchain-openai requests pydantic
!pip install -q langchain-groq
!pip install -q sentence-transformers
!pip install -q chromadb rank-bm25

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
t

## 2. Configuration des clés API et modèles

In [20]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Clé chargée :", "GROQ_API_KEY" in os.environ)

Clé chargée : True


## 3. Vérification de l'environnement

In [3]:
import langgraph, langchain, langchain_openai
import importlib.metadata

print("langgraph       :", importlib.metadata.version("langgraph"))
print("langchain       :", langchain.__version__)
print("langchain-openai:", langchain_openai.__version__)

langgraph       : 1.2.19
langchain       : 1.4.8
langchain-openai: 1.8.0


## 4. Importation des bibliothèques

In [4]:
import csv, json, operator, os, re, sys, time
from pathlib import Path
from typing import Annotated, Literal, TypedDict

import requests
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from langchain_openai import ChatOpenAI

print("Tous les imports OK")

Tous les imports OK


## 5. Tests unitaires des briques du pipeline

### 5.1 Validation du schéma structuré LLM (Pydantic)

In [5]:
from pydantic import BaseModel, Field

class Protocol(BaseModel):
    objective: str
    inclusion_criteria: list[str]
    exclusion_criteria: list[str]
    year_min: int = Field(description="année minimale de publication")

# Nouveau code (Groq)
from langchain_groq import ChatGroq
llm = ChatGroq(
    # model="openai/gpt-oss-120b",  # Un des modèles gratuits les plus performants
    model = "qwen/qwen3.8-27b",
    timeout=90,           # 90s max par requête
    max_retries=1,        # un seul retry (évite le backoff infini)
    temperature=0,
)

try:
    p = llm.with_structured_output(Protocol, method="json_schema").invoke(
        "Prépare un protocole de revue systématique sur : 'effet du café sur la cognition'."
    )
    print("json_mode OK")
    print(p.model_dump_json(indent=2))
except Exception as e:
    print("json_mode échoue :", type(e).__name__, e)

json_mode OK
{
  "objective": "Évaluer l'effet de la consommation de café sur les fonctions cognitives chez l'adulte sain.",
  "inclusion_criteria": [
    "Essais randomisés contrôlés (ERC)",
    "Études observationnelles prospectives",
    "Participants adultes (18 ans et plus)",
    "Intervention : consommation de café (caféine ou extrait de café)",
    "Mesure de la cognition (mémoire, attention, vitesse de traitement, fonctions exécutives)",
    "Publication en anglais, français ou espagnol",
    "Publication entre 2010 et 2024"
  ],
  "exclusion_criteria": [
    "Études sur les animaux",
    "Études in vitro",
    "Éditoriaux",
    "Lettres à l'éditeur",
    "Études non contrôlées",
    "Études sur les populations spécifiques non représentatives (ex : patients en soins intensifs)"
  ],
  "year_min": 2010
}


### 5.2 Test de récupération sur l'API OpenAlex

In [6]:
def _abstract(inv):
    if not inv:
        return ""
    pos = {i: w for w, idxs in inv.items() for i in idxs}
    return " ".join(pos[i] for i in sorted(pos))

def search_openalex(q, n=10, year_min=2020):
    r = requests.get(
        "https://api.openalex.org/works",
        params={"search": q, "per-page": n,
                "filter": f"from_publication_date:{year_min}-01-01,has_abstract:true",
                "mailto": "ton.email@example.com"},  # pool poli
        timeout=30,
    )
    r.raise_for_status()
    return [{"title": w.get("title"), "year": w.get("publication_year"),
             "abstract": _abstract(w.get("abstract_inverted_index"))}
            for w in r.json()["results"]]

hits = search_openalex("coffee cognition", n=3)
print(f"✅ {len(hits)} résultats")
for h in hits:
    print(f"  {h['year']} — {h['title'][:80]}")

✅ 3 résultats
  2021 — Bulletproof Coffee and Cognition
  2026 — Coffee and mental health: Pharmacological effects on cognition, anxiety, depress
  2024 — Impact of coffee-derived chlorogenic acid on cognition: a systematic review and 


## 6. Architecture du graphe LangGraph et RAG hybride (review_graph.py)

In [11]:
%%writefile /content/review_graph.py
"""Pipeline agentique de revue systématique — Colab/Groq + RAG Hybride (Chroma + BM25) multi-sujets."""
import csv
import hashlib
import json
import operator
import os
import re
import threading
import time
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Annotated, Any, Literal, TypedDict

import requests
from groq import RateLimitError
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel, Field

# ============================ BLOC 1 : LLM ============================
MODEL_A = os.environ.get("MODEL_A", "openai/gpt-oss-120b")
MODEL_B = os.environ.get("MODEL_B", "qwen/qwen3.8-27b")
DEFAULT_N_PER_QUERY = int(os.environ.get("N_PER_QUERY", "2"))


def make_llm(model, **kw):
    # max_retries=0 : c'est NOTRE retry (bloc 2) qui gère les 429
    return ChatGroq(model=model, temperature=0, timeout=90, max_retries=0, **kw)


# Génération longue (planner, synthèse, critic) : reasoning bas, pas de plafond serré
llm = make_llm(MODEL_A, reasoning_effort="low")
# Screening / extraction : sorties courtes => on plafonne (max_tokens compte dans le TPM demandé)
llm_screen_a = make_llm(MODEL_A, reasoning_effort="low", max_tokens=500)
llm_screen_b = make_llm(MODEL_B, max_tokens=600)  # marge suffisante si Qwen émet <think>...</think>


# --- Rate limiter ---
class TokenBucket:
    def __init__(self, tpm=6000):
        self.tpm = tpm
        self.tokens = tpm
        self.last = time.time()
        self.lock = threading.Lock()

    def consume(self, n_tokens):
        with self.lock:
            now = time.time()
            self.tokens = min(self.tpm, self.tokens + (now - self.last) * (self.tpm / 60))
            self.last = now
            if self.tokens < n_tokens:
                wait = (n_tokens - self.tokens) * 60 / self.tpm
                print(f"    ⏳ TPM budget épuisé, attente {wait:.1f}s")
                time.sleep(wait)
                self.tokens = 0
            else:
                self.tokens -= n_tokens


bucket = TokenBucket(tpm=6000)  # 75 % de la limite, la fenêtre Groq est glissante

# ============================ BLOC 2 : retry + cache ============================
_WAIT_RE = re.compile(r"try again in\s+(?:(\d+)m(?!s))?\s*(?:([\d.]+)(ms|s))?", re.I)


def _retry_after(msg: str) -> float:
    m = _WAIT_RE.search(msg)
    if not m or not (m.group(1) or m.group(2)):
        return 8.0
    secs = float(m.group(2) or 0)
    if m.group(3) == "ms":
        secs /= 1000
    return int(m.group(1) or 0) * 60 + secs + 1.0


def call_llm(runnable, prompt, est_tokens, tries=6):
    """Appel LLM avec budget TPM + attente exacte demandée par Groq en cas de 429."""
    for attempt in range(1, tries + 1):
        bucket.consume(est_tokens)
        try:
            return runnable.invoke(prompt)
        except RateLimitError as e:
            wait = _retry_after(str(e))
            if wait > 120:  # quota JOURNALIER (TPD) : inutile d'attendre
                raise RuntimeError(f"Quota journalier Groq atteint : {e}") from e
            print(f"    ⏳ 429 Groq : attente {wait:.0f}s (essai {attempt}/{tries})")
            time.sleep(wait)
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                wait = _retry_after(str(e))
                if wait > 120:
                    raise RuntimeError(f"Quota journalier Groq atteint : {e}") from e
                print(f"    ⏳ 429 (générique) : attente {wait:.0f}s (essai {attempt}/{tries})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Rate limit persistant après plusieurs essais")


CACHE_FILE = Path("/content/llm_cache.json")
CACHE_TAG = os.environ.get("CACHE_TAG", "v1")
try:
    _cache = json.loads(CACHE_FILE.read_text(encoding="utf-8")) if CACHE_FILE.exists() else {}
except Exception:
    _cache = {}


def _topic_tag(question: str) -> str:
    """Isole automatiquement le cache par sujet tout en conservant le cache intra-sujet."""
    q_hash = hashlib.sha1((question or "").strip().lower().encode("utf-8")).hexdigest()[:8]
    return f"{CACHE_TAG}:{q_hash}"


def cached_call(key: str, fn, namespace: str = CACHE_TAG):
    k = f"{namespace}|{key}"
    if k not in _cache:
        _cache[k] = fn()
        CACHE_FILE.write_text(json.dumps(_cache, ensure_ascii=False), encoding="utf-8")
    return _cache[k]


def compact_protocol(p: dict) -> str:
    """Critères seuls (sans 'objective') : ~40 % de tokens en moins par appel de screening."""
    return ("Include if: " + "; ".join(p["inclusion_criteria"]) +
            "\nExclude if: " + "; ".join(p["exclusion_criteria"]))


# ============================ RAG HYBRIDE : Chroma + BM25 ============================
_embed_model = None
_RAG_INDEXES: dict[str, dict[str, Any]] = {}


def _get_embed_model():
    """Charge sentence-transformers une seule fois en local (0 quota API)."""
    global _embed_model
    if _embed_model is None:
        from sentence_transformers import SentenceTransformer
        print("    🔧 Chargement du modèle d'embedding local (all-MiniLM-L6-v2)...")
        _embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        print("    ✅ Modèle d'embedding chargé")
    return _embed_model


def _tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", (text or "").lower())


def build_index(records):
    """Construit un index RAG hybride : base vectorielle Chroma + index lexical BM25."""
    if not records:
        return None
    import chromadb
    from rank_bm25 import BM25Okapi

    model = _get_embed_model()
    texts = [f"{r['title']}. {r['abstract']}" for r in records]
    embs = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

    # 1. Index dense : ChromaDB (similarité cosinus)
    client = chromadb.EphemeralClient()
    col_name = f"prisma_rag_{int(time.time() * 1000)}"
    collection = client.create_collection(
        name=col_name,
        metadata={"hnsw:space": "cosine"},
    )
    collection.add(
        ids=[str(r["id"]) for r in records],
        embeddings=embs.tolist(),
        documents=texts,
        metadatas=[
            {
                "id": int(r["id"]),
                "title": (r["title"] or "")[:200],
                "year": int(r["year"] or 0),
                "source": r.get("source", ""),
            }
            for r in records
        ],
    )

    # 2. Index lexical : BM25Okapi
    bm25 = BM25Okapi([_tokenize(t) for t in texts])

    _RAG_INDEXES[col_name] = {
        "client": client,
        "collection": collection,
        "bm25": bm25,
        "records": records,
    }
    return {
        "index_id": col_name,
        "n_docs": len(records),
        "dim": int(embs.shape[1]),
        "backend": "chroma+bm25",
    }


def retrieve_top_k(query: str, index, k: int = 2, alpha: float = 0.6):
    """Recherche hybride (alpha * Chroma cosinus + (1 - alpha) * BM25 normalisé)."""
    if not index or not query:
        return []
    store = _RAG_INDEXES.get(index["index_id"]) if isinstance(index, dict) else None
    if not store:
        return []

    records = store["records"]
    model = _get_embed_model()
    q_emb = model.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]

    # Scores denses via Chroma
    n_res = min(len(records), max(k * 3, 10))
    chroma_res = store["collection"].query(
        query_embeddings=[q_emb.tolist()],
        n_results=n_res,
    )
    dense_scores = {
        int(doc_id): max(0.0, 1.0 - float(dist))
        for doc_id, dist in zip(chroma_res["ids"][0], chroma_res["distances"][0])
    }

    # Scores lexicaux via BM25
    raw_bm25 = store["bm25"].get_scores(_tokenize(query))
    max_bm25 = float(max(raw_bm25)) if len(raw_bm25) > 0 else 0.0
    bm25_scores = {
        rec["id"]: (float(score) / max_bm25 if max_bm25 > 0 else 0.0)
        for rec, score in zip(records, raw_bm25)
    }

    scored = []
    for rec in records:
        rid = rec["id"]
        d_sc = dense_scores.get(rid, 0.0)
        b_sc = bm25_scores.get(rid, 0.0)
        h_sc = alpha * d_sc + (1.0 - alpha) * b_sc
        scored.append((h_sc, d_sc, b_sc, rec))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [
        {
            "id": rec["id"],
            "title": rec["title"],
            "abstract": rec["abstract"],
            "score": float(h_sc),
            "dense_score": float(d_sc),
            "bm25_score": float(b_sc),
        }
        for h_sc, d_sc, b_sc, rec in scored[:k]
    ]


# --- Clé OpenAlex & gestion dynamique du dossier de sortie ---
OPENALEX_KEY = os.environ.get("OPENALEX_API_KEY", "")
OUT = Path("runs") / time.strftime("%Y%m%d_%H%M%S")
OUT.mkdir(parents=True, exist_ok=True)


def _get_out_dir(state: dict | None = None) -> Path:
    if state and state.get("out_dir"):
        p = Path(state["out_dir"])
        p.mkdir(parents=True, exist_ok=True)
        return p
    OUT.mkdir(parents=True, exist_ok=True)
    return OUT


class ReviewState(TypedDict, total=False):
    question: str
    n_per_query: int
    out_dir: str
    human_feedback: str
    protocol: dict
    queries: list[str]
    records: list[dict]
    index: Any
    screening: list[dict]
    included: list[dict]
    extraction: list[dict]
    report: str
    critic_issues: list[str]
    revisions: int
    prisma: dict
    log: Annotated[list[dict], operator.add]


def trace(agent: str, event: str, data=None, state: dict | None = None) -> dict:
    entry = {"t": time.time(), "agent": agent, "event": event, "data": data}
    out_dir = _get_out_dir(state)
    with open(out_dir / "trace.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False, default=str) + "\n")
    return entry


class Protocol(BaseModel):
    objective: str
    inclusion_criteria: list[str]
    exclusion_criteria: list[str]
    year_min: int = Field(default=2018, description="minimum publication year")


class Queries(BaseModel):
    queries: list[str] = Field(description="4 to 5 short English keyword queries covering all facets of the question")


class ScreenDecision(BaseModel):
    decision: Literal["include", "exclude"]
    criterion: str = Field(description="criterion used, max 10 words")
    rationale: str = Field(description="max 20 words")


class Extraction(BaseModel):
    study_design: str
    data_or_population: str
    method: str
    key_results: str
    limitations: str
    quality_score: int = Field(ge=0, le=5)
    quality_rationale: str


class CriticVerdict(BaseModel):
    ok: bool
    issues: list[str]


def _abstract(inv):
    if not inv:
        return ""
    pos = {i: w for w, idxs in inv.items() for i in idxs}
    return " ".join(pos[i] for i in sorted(pos))


def search_openalex(q, n=DEFAULT_N_PER_QUERY, year_min=2018, max_retries=3):
    url = "https://api.openalex.org/works"
    headers = {"User-Agent": "SystematicReviewAgent/1.0 (mailto:colab-research@example.org)"}
    if OPENALEX_KEY:
        headers["Authorization"] = f"Bearer {OPENALEX_KEY}"
    current_q = q
    for attempt in range(max_retries):
        params = {
            "search": current_q,
            "per-page": n,
            "filter": f"from_publication_date:{year_min}-01-01",
        }
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            if r.status_code in (429, 500, 502, 503, 504):
                if r.status_code == 504 and len(current_q.split()) > 3:
                    current_q = " ".join(current_q.split()[:3])
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            results = r.json().get("results", [])
            print(f"    OpenAlex '{current_q[:50]}' → {len(results)} hits")
            return [{"source": "openalex", "title": w.get("title") or "", "doi": w.get("doi"),
                     "year": w.get("publication_year"), "url": w.get("id"),
                     "abstract": _abstract(w.get("abstract_inverted_index"))}
                    for w in results]
        except requests.exceptions.RequestException:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
    print(f"    ❌ OpenAlex abandonné : {q[:60]}")
    return []


def _simplify_query(q: str) -> str:
    """Réduit une requête booléenne complexe à des mots-clés simples tout en préservant les acronymes techniques (ex: GLiNER, LoRA, NER)."""
    q = re.sub(r'[()"\']', ' ', q)
    q = re.sub(r'\b(AND|OR|NOT)\b', ' ', q, flags=re.IGNORECASE)
    q = re.sub(r'publication_year\s*:\s*[<>=]+\s*\d+', ' ', q)
    q = re.sub(r'\byear\s*[<>=]+\s*\d+', ' ', q)
    words = [w for w in q.split() if len(w) >= 3][:8]
    return " ".join(words)


def search_arxiv(q, n=DEFAULT_N_PER_QUERY, year_min=2018, max_retries=3):
    """Recherche sur l'API Atom officielle d'arXiv."""
    url = "https://export.arxiv.org/api/query"
    params = {
        "search_query": f"all:{q}",
        "start": 0,
        "max_results": n,
        "sortBy": "relevance",
        "sortOrder": "descending",
    }
    headers = {"User-Agent": "SystematicReviewAgent/1.0 (Colab)"}
    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom",
    }
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            root = ET.fromstring(r.text)
            hits = []
            for entry in root.findall("atom:entry", ns):
                published = entry.findtext("atom:published", "", ns)
                year = int(published[:4]) if published[:4].isdigit() else None
                if year and year < year_min:
                    continue
                title = " ".join(entry.findtext("atom:title", "", ns).split())
                abstract = " ".join(entry.findtext("atom:summary", "", ns).split())
                entry_url = entry.findtext("atom:id", "", ns).strip()
                doi_raw = entry.findtext("arxiv:doi", "", ns).strip()
                doi = f"https://doi.org/{doi_raw}" if doi_raw else None
                hits.append({
                    "source": "arxiv",
                    "title": title,
                    "doi": doi,
                    "year": year,
                    "url": entry_url,
                    "abstract": abstract,
                })
            print(f"    arXiv    '{q[:50]}' → {len(hits)} hits")
            return hits
        except (requests.exceptions.RequestException, ET.ParseError):
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
    print(f"    ❌ arXiv abandonné : {q[:60]}")
    return []


def planner(state):
    # Initialise un dossier dédié au sujet courant si non défini
    out_dir = state.get("out_dir")
    if not out_dir:
        slug = re.sub(r"[^a-z0-9]+", "_", state["question"].lower()).strip("_")[:35]
        p = Path("runs") / f"{time.strftime('%Y%m%d_%H%M%S')}_{slug}"
        p.mkdir(parents=True, exist_ok=True)
        out_dir = str(p)

    state_with_dir = {**state, "out_dir": out_dir}
    feedback = state.get("human_feedback", "")
    prompt = (
        f"You are preparing a systematic review (PRISMA 2020).\n"
        f"Research question: {state['question']}\n"
        f"User feedback to integrate: {feedback or 'none'}\n"
        "Produce the protocol. IMPORTANT: You must define a realistic integer value "
        "for the required 'year_min' field (e.g., 2018 or 2020) under any circumstances.\n"
        "IMPORTANT:\n"
        "- Write everything in ENGLISH.\n"
        "- Adapt the criteria strictly to the research domain (e.g., empirical benchmarks/datasets/models "
        "for computer science/NLP, or clinical/observational cohorts for biomedical topics).\n"
        "- Specify ONLY OpenAlex and arXiv as searched databases (do NOT mention PubMed or other unqueried sources)."
    )
    structured_planner = llm.with_structured_output(Protocol, method="json_schema")
    protocol = call_llm(structured_planner, prompt, est_tokens=1500).model_dump()
    if not protocol.get("year_min"):
        protocol["year_min"] = 2018
    return {
        "out_dir": out_dir,
        "protocol": protocol,
        "log": [trace("planner", "protocol", protocol, state_with_dir)],
    }


def human_protocol(state):
    answer = interrupt({"validate_protocol": state["protocol"],
                        "reply": "ok, or your comments to revise"})
    if str(answer).strip().lower() == "ok":
        return Command(goto="search_strategy",
                       update={"log": [trace("human", "protocol_approved", state=state)]})
    return Command(goto="planner", update={"human_feedback": str(answer),
                                           "log": [trace("human", "protocol_feedback", answer, state)]})


def search_strategy(state):
    prompt = (
        f"Research question: {state['question']}\n"
        f"Protocol: {json.dumps(state['protocol'], ensure_ascii=False)}\n"
        "Generate 4 to 5 targeted search queries for OpenAlex and arXiv.\n"
        "IMPORTANT:\n"
        "- Write queries in ENGLISH only.\n"
        "- Use SHORT keyword phrases (3-6 words), NO Boolean operators (AND/OR/NOT) and NO parentheses.\n"
        "- Do NOT include year filters or language filters (handled separately).\n"
        "- Cover all distinct methods, models, and evaluation dimensions mentioned in the research question "
        "(e.g., one query per model family or evaluation aspect)."
    )
    structured_search = llm.with_structured_output(Queries, method="json_schema")
    queries = call_llm(structured_search, prompt, est_tokens=1500).queries
    return {"queries": queries, "log": [trace("search_strategy", "queries", queries, state)]}


def retrieval(state):
    raw = []
    year_min = state["protocol"].get("year_min") or 2018
    n_hits = int(state.get("n_per_query") or DEFAULT_N_PER_QUERY)
    for q in state["queries"]:
        simple = _simplify_query(q) or q
        found_oa = search_openalex(simple, n=n_hits, year_min=year_min)
        found_ax = search_arxiv(simple, n=n_hits, year_min=year_min)
        found = found_oa + found_ax
        trace("retrieval", "query_hits", {
            "query": q, "simplified": simple, "openalex": len(found_oa), "arxiv": len(found_ax)
        }, state)
        raw += found
        time.sleep(0.3)
    seen, records = set(), []
    for r in raw:
        if not r["abstract"] or len(r["abstract"]) < 100:
            continue
        key = (r["doi"] or re.sub(r"\W+", "", (r["title"] or "").lower()))
        if key and key not in seen:
            seen.add(key)
            records.append({**r, "id": len(records) + 1})

    print("    🔍 Construction de l'index RAG hybride (Chroma + BM25)...")
    index = build_index(records)
    if index:
        print(f"    ✅ Index RAG prêt : {index['n_docs']} documents (Chroma dim={index['dim']} + BM25)")

    prisma = {"identified": len(raw), "duplicates_removed": len(raw) - len(records)}
    return {
        "records": records,
        "index": index,
        "prisma": prisma,
        "log": [
            trace("retrieval", "done", prisma, state),
            trace("retrieval", "index_built", index, state),
        ],
    }


REVIEWER_STYLES = {
    "A": "You are a STRICT reviewer: exclude if any inclusion criterion is uncertain.",
    "B": "You are an INCLUSIVE reviewer: include if in doubt — full text will decide.",
}


# ============================ BLOC 3 : screening + extraction ============================
def screening(state):
    runnable_a = llm_screen_a.with_structured_output(ScreenDecision, method="json_schema")
    runnable_b = llm_screen_b.with_structured_output(ScreenDecision, method="json_schema")

    ns = _topic_tag(state["question"])
    proto = compact_protocol(state["protocol"])
    results, included = [], []

    for i, rec in enumerate(state["records"], 1):
        votes = {}
        # Relecteur A
        prompt_a = (
            f"{REVIEWER_STYLES['A']}\n{proto}\n"
            f"Title: {rec['title']}\nAbstract: {rec['abstract'][:600]}"
        )
        votes["A"] = cached_call(
            f"screen|A|{rec['title']}",
            lambda p=prompt_a: call_llm(runnable_a, p, len(p) // 3 + 250).model_dump(),
            namespace=ns,
        )

        # Relecteur B (json_schema + fallback sécurisé pour Qwen/Llama)
        prompt_b = (
            f"{REVIEWER_STYLES['B']}\n{proto}\n"
            f"Title: {rec['title']}\nAbstract: {rec['abstract'][:600]}\n"
            'Return valid JSON with exact keys: "decision" ("include" or "exclude"), '
            '"criterion" (max 10 words), "rationale" (max 20 words).'
        )

        def call_b_with_fallback(p=prompt_b):
            try:
                return call_llm(runnable_b, p, len(p) // 3 + 250).model_dump()
            except (RateLimitError, RuntimeError):
                raise  # Laisse remonter les limites de taux / quotas journaliers
            except Exception:
                raw = call_llm(llm_screen_b, p, len(p) // 3 + 250).content
                # Supprime les réflexions internes <think>...</think> éventuelles (Qwen)
                clean = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
                try:
                    match = re.search(r"\{.*\}", clean, re.DOTALL)
                    data = json.loads(match.group(0)) if match else json.loads(clean)
                    dec_val = str(data.get("decision") or data.get("answer") or "").lower()
                    dec = "include" if ("include" in dec_val or data.get("includes") is True) else "exclude"
                    crit = data.get("criterion") or data.get("reason") or "Protocol criterion"
                    rat = data.get("rationale") or data.get("reason") or "Parsed from JSON fallback"
                    return {"decision": dec, "criterion": str(crit)[:80], "rationale": str(rat)[:150]}
                except Exception:
                    dec = "include" if re.search(r"\binclude\b", clean, re.IGNORECASE) else "exclude"
                    return {"decision": dec, "criterion": "Fallback parsing", "rationale": "Extracted from text"}

        votes["B"] = cached_call(
            f"screen|B|{rec['title']}",
            call_b_with_fallback,
            namespace=ns,
        )

        a, b = votes["A"]["decision"], votes["B"]["decision"]
        final = "include" if "include" in (a, b) else "exclude"
        results.append({
            "id": rec["id"], "title": rec["title"],
            "A": votes["A"], "B": votes["B"],
            "conflict": a != b, "final": final,
        })
        if final == "include":
            included.append(rec)
        print(f"    [{i}/{len(state['records'])}] {final:7s} {rec['title'][:60]}")

    prisma = {
        **state["prisma"],
        "screened": len(state["records"]),
        "excluded": len(state["records"]) - len(included),
        "included": len(included),
    }
    out_dir = _get_out_dir(state)
    with open(out_dir / "screening.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    return {
        "screening": results, "included": included, "prisma": prisma,
        "log": [trace("screening", "done", prisma, state)],
    }


def human_screening(state):
    conflicts = [s["title"] for s in state["screening"] if s["conflict"]][:5]
    answer = interrupt({"prisma": state["prisma"], "conflict_examples": conflicts,
                        "reply": "ok to continue"})
    return {"log": [trace("human", "screening_reviewed", answer, state)]}


def extraction(state):
    if not state.get("included"):
        return {"extraction": [], "log": [trace("extraction", "skipped_no_included", state=state)]}
    ns = _topic_tag(state["question"])
    structured = llm_screen_a.with_structured_output(Extraction, method="json_schema")
    table = []
    for rec in state["included"]:
        rag_hits = retrieve_top_k(f"{state['question']} {rec['title']}", state.get("index"), k=2)
        rag_context = ""
        if rag_hits:
            rag_context = "\nRelated context (Chroma+BM25 cross-references): " + " | ".join(
                f"[{h['id']}] {h['title'][:75]} (hybrid={h['score']:.2f}, dense={h['dense_score']:.2f}, bm25={h['bm25_score']:.2f})"
                for h in rag_hits
            )
            print(
                f"    🔍 RAG (Chroma+BM25) pour '{rec['title'][:38]}' : "
                f"{[(h['id'], round(h['score'], 2)) for h in rag_hits]}"
            )

        prompt = (
            "Extract structured data relevant to the research question from this study. "
            "Answer in ENGLISH, keep each field under 25 words.\n"
            f"Research question: {state['question']}\n"
            f"Title: {rec['title']}\nAbstract: {rec['abstract'][:600]}{rag_context}"
        )
        row = cached_call(
            f"extract-rag|{rec['title']}",
            lambda p=prompt: call_llm(structured, p, len(p) // 3 + 400).model_dump(),
            namespace=ns,
        )
        table.append({"id": rec["id"], "title": rec["title"], "year": rec["year"], **row})
    return {"extraction": table, "log": [trace("extraction", "done", {"n": len(table)}, state)]}


def _build_references_section(included: list[dict]) -> str:
    """Génère une bibliographie vérifiée à partir des études incluses."""
    lines = ["\n## References\n"]
    for r in included:
        link = r.get("doi") or r.get("url") or ""
        src = r.get("source", "").upper()
        year = r.get("year") or "n.d."
        ref = f"- **[{r['id']}]** {r['title']} ({year}). *{src}*."
        if link:
            ref += f" {link}"
        lines.append(ref)
    return "\n".join(lines) + "\n"


def _deliver_outputs(state: dict, report: str):
    """Écrit et livre sur disque le rapport Markdown, le tableau CSV et les stats PRISMA."""
    out_dir = _get_out_dir(state)
    report_path = out_dir / "report.md"
    colab_copy = Path("/content/systematic_review_report.md")
    report_path.write_text(report, encoding="utf-8")
    colab_copy.write_text(report, encoding="utf-8")

    if state.get("prisma"):
        (out_dir / "prisma.json").write_text(
            json.dumps(state["prisma"], ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    extraction_rows = state.get("extraction") or []
    if extraction_rows:
        csv_path = out_dir / "extraction.csv"
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(extraction_rows[0].keys()))
            writer.writeheader()
            writer.writerows(extraction_rows)
    print(f"    📄 Synthèse livrée : {report_path} (et {colab_copy})")


def synthesis(state):
    if not state.get("extraction"):
        empty_report = "# Systematic Literature Review\n\nNo included studies — cannot synthesize."
        _deliver_outputs(state, empty_report)
        return {"report": empty_report,
                "revisions": state.get("revisions", 0) + 1,
                "log": [trace("synthesis", "skipped_no_extraction", state=state)]}

    issues = state.get("critic_issues") or []
    prisma = state.get("prisma") or {}
    n_included = len(state["extraction"])
    global_rag = retrieve_top_k(state["question"], state.get("index"), k=min(5, n_included))
    rag_summary = ", ".join(f"[{h['id']}] (score={h['score']:.2f})" for h in global_rag)

    prompt = (
        "Write a complete, publication-grade systematic literature review in ENGLISH (Markdown).\n"
        f"Research question: {state['question']}\n"
        f"Exact number of included studies: {n_included}\n"
        f"PRISMA 2020 counts: {json.dumps(prisma, ensure_ascii=False)}\n"
        f"Top RAG-ranked studies (Chroma + BM25): {rag_summary}\n"
        f"Evidence table (quality_score is on a 0 to 5 scale): {json.dumps(state['extraction'], ensure_ascii=False)}\n\n"
        "Required structure:\n"
        "# Systematic Literature Review: <concise title>\n"
        "## 1. Executive Summary\n"
        "## 2. Methodology & PRISMA Flow (databases searched: OpenAlex and arXiv only; report exact PRISMA counts)\n"
        "## 3. Characteristics & Quality of Included Studies (include a Markdown comparison table citing [id] and quality_score/5)\n"
        "## 4. Thematic & Comparative Synthesis (organize around the key methods/dimensions of the research question, cite every claim with [id])\n"
        "## 5. Limitations & Future Directions\n\n"
        "Rules:\n"
        "- State the exact count of included studies consistently across all sections.\n"
        "- Cite ONLY valid study IDs using [id] matching the evidence table.\n"
        "- Do NOT invent a References section (it will be appended automatically).\n"
        f"- Issues to fix from previous critic review: {issues or 'none'}"
    )
    draft = call_llm(llm, prompt, est_tokens=len(prompt) // 3 + 1800).content
    draft_clean = re.split(r"\n##\s*References\b", draft, flags=re.IGNORECASE)[0].rstrip()
    full_report = draft_clean + "\n" + _build_references_section(state.get("included") or [])

    _deliver_outputs(state, full_report)
    rev = state.get("revisions", 0) + 1
    return {
        "report": full_report,
        "revisions": rev,
        "log": [trace("synthesis", "draft_delivered", {"revision": rev, "chars": len(full_report)}, state)],
    }


def critic(state):
    if not state.get("extraction"):
        return {"critic_issues": [], "log": [trace("critic", "skipped_empty", state=state)]}
    valid = {str(r["id"]) for r in state["included"]}
    body = re.split(r"\n##\s*References\b", state["report"], flags=re.IGNORECASE)[0]
    cited = set(re.findall(r"\[(\d+)\]", body))
    issues = [f"Unknown citation [{c}]" for c in cited - valid]
    critic_prompt = (
        f"Protocol: {json.dumps(state['protocol'], ensure_ascii=False)}\n"
        f"Included studies count: {len(state['included'])}\n"
        f"Report: {body}\n"
        "Check ONLY fixable factual errors in the report: "
        "1) consistency with the protocol and exact PRISMA counts, "
        "2) unsupported claims or overstatements, "
        "3) accurate quality_score reporting (scale is 0 to 5). "
        "Note: inline numeric citations [id] matching the evidence table are the required format (do NOT flag [id] formatting), "
        "and do NOT flag inherent small sample size of the retrieved corpus."
    )
    structured_critic = llm.with_structured_output(CriticVerdict, method="json_schema")
    verdict = call_llm(structured_critic, critic_prompt, est_tokens=1500)
    issues += verdict.issues
    return {"critic_issues": issues, "log": [trace("critic", "verdict", issues, state)]}


def route_after_critic(state):
    if not state.get("extraction"):
        return "end"
    return "end" if not state["critic_issues"] or state["revisions"] >= 2 else "synthesis"


def build_graph():
    g = StateGraph(ReviewState)
    for name, fn in [("planner", planner), ("human_protocol", human_protocol),
                     ("search_strategy", search_strategy), ("retrieval", retrieval),
                     ("screening", screening), ("human_screening", human_screening),
                     ("extraction", extraction), ("synthesis", synthesis), ("critic", critic)]:
        g.add_node(name, fn)
    g.add_edge(START, "planner")
    g.add_edge("planner", "human_protocol")
    g.add_edge("search_strategy", "retrieval")
    g.add_edge("retrieval", "screening")
    g.add_edge("screening", "human_screening")
    g.add_edge("human_screening", "extraction")
    g.add_edge("extraction", "synthesis")
    g.add_edge("synthesis", "critic")
    g.add_conditional_edges("critic", route_after_critic, {"synthesis": "synthesis", "end": END})
    return g.compile(checkpointer=MemorySaver())

Overwriting /content/review_graph.py


### 6.1 Chargement du module

In [12]:
import importlib, sys
sys.path.insert(0, "/content")
import review_graph
importlib.reload(review_graph)
print("Module rechargé")

Module rechargé


## 7. Exécution complète de la revue systématique (Sujet : NER)

In [13]:
import importlib
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import cohen_kappa_score

# 1. Vérification minimale de l'environnement Colab
assert os.environ.get("GROQ_API_KEY"), "Définis os.environ['GROQ_API_KEY'] avant de lancer le test."

sys.path.insert(0, "/content")
import review_graph
importlib.reload(review_graph)


def run_auto(
    question: str,
    responses: list[str] | None = None,
    n_per_query: int = 3,
    reset_topic_cache: bool = False,
) -> dict:
    """Exécute le graphe de revue systématique de bout en bout."""
    responses = responses or ["ok", "ok", "ok"]

    if reset_topic_cache:
        ns = review_graph._topic_tag(question)
        review_graph._cache = {
            k: v for k, v in review_graph._cache.items() if not k.startswith(f"{ns}|")
        }
        if review_graph.CACHE_FILE.exists():
            review_graph.CACHE_FILE.write_text(
                json.dumps(review_graph._cache, ensure_ascii=False), encoding="utf-8"
            )
        print(f"🧹 Cache nettoyé pour le sujet courant ({ns})")

    app = review_graph.build_graph()
    cfg = {"configurable": {"thread_id": f"e2e-{int(time.time())}"}}
    result = app.invoke({"question": question, "n_per_query": n_per_query}, cfg)

    step = 0
    while result.get("__interrupt__"):
        payload = result["__interrupt__"][0].value
        print(f"\n--- INTERRUPT #{step + 1} ---")
        print(json.dumps(payload, ensure_ascii=False, indent=2)[:800])
        reply = responses[step] if step < len(responses) else "ok"
        print(f"→ réponse auto : {reply!r}")
        result = app.invoke(review_graph.Command(resume=reply), cfg)
        step += 1

    return result


# 2. Lancement du test de bout en bout sur le sujet NER
question_ner = (
    "Comparison of information extraction approaches (fine-tuned encoders, GLiNER, "
    "prompted LLMs, LoRA fine-tuned LLMs) for named entity recognition: "
    "accuracy, cost, and out-of-domain generalization"
)

print("🚀 Lancement du pipeline complet...")
t0 = time.time()
result = run_auto(
    question_ner,
    responses=["ok", "ok"],
    n_per_query=2,
    reset_topic_cache=True,
)
elapsed = time.time() - t0

# 3. Audit automatique des résultats et métriques d'évaluation
out_dir = Path(result["out_dir"])
print("\n" + "=" * 65)
print(f"⏱️ Temps total           : {elapsed:.1f}s")
print(f"📁 Dossier de sortie     : {out_dir}")
print(f"📄 Fichiers générés      : {[p.name for p in out_dir.iterdir()]}")
print(f"📊 Statistiques PRISMA   : {result.get('prisma')}")
print(f"✅ Études incluses       : {len(result.get('included', []))}")
print(f"🧾 Extractions           : {len(result.get('extraction', []))}")
print(f"🔁 Révisions critic      : {result.get('revisions')}")
print(f"⚠️ Issues restantes      : {result.get('critic_issues')}")

# Accord inter-relecteurs (A vs B) + Kappa de Cohen
screen_path = out_dir / "screening.json"
if screen_path.exists():
    screen_data = json.loads(screen_path.read_text(encoding="utf-8"))
    if screen_data:
        agreements = sum(1 for s in screen_data if not s["conflict"])
        votes_a = [s["A"]["decision"] for s in screen_data]
        votes_b = [s["B"]["decision"] for s in screen_data]
        kappa = (
            cohen_kappa_score(votes_a, votes_b)
            if len(set(votes_a + votes_b)) > 1
            else 1.0
        )
        print(f"🤝 Accord brut A / B     : {agreements}/{len(screen_data)} ({agreements / len(screen_data):.1%})")
        print(f"📐 Kappa de Cohen (A/B)  : {kappa:.3f}")

# Aperçu du tableau d'extraction
csv_path = out_dir / "extraction.csv"
if csv_path.exists():
    print("\n--- TABLEAU D'EXTRACTION ---")
    df_ext = pd.read_csv(csv_path)
    display(df_ext[["id", "year", "title", "method", "quality_score"]])

# Affichage de la synthèse finale
print("\n--- RAPPORT DE SYNTHÈSE ---")
display(Markdown(result.get("report") or "# Aucun rapport généré"))

🚀 Lancement du pipeline complet...
🧹 Cache nettoyé pour le sujet courant (v1:ed6f0e3c)

--- INTERRUPT #1 ---
{
  "validate_protocol": {
    "objective": "To systematically compare information extraction approaches—fine‑tuned encoder models (e.g., BERT, RoBERTa), GLiNER, prompted large language models (LLMs), and LoRA‑fine‑tuned LLMs—for named entity recognition (NER) in terms of accuracy, computational cost, and out‑of‑domain generalization.",
    "inclusion_criteria": [
      "Primary research articles that evaluate at least one of the four target NER approaches on a publicly available benchmark dataset.",
      "Studies reporting quantitative performance metrics for NER (e.g., precision, recall, F1‑score) and, when available, computational cost measures (e.g., inference latency, GPU hours, monetary cost).",
      "Experiments that assess out‑of‑domain or cross‑domain generalization (e.g., train
→ réponse auto : 'ok'
    OpenAlex 'fine tuned encoder NER' → 2 hits
    arXiv    'fine tu

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

    ✅ Modèle d'embedding chargé
    ✅ Index RAG prêt : 19 documents (Chroma dim=384 + BM25)
    [1/19] include Uzbek Named Entity Recognition: In-Domain Fine-Tuned XLM-R B
    [2/19] include Domain-Adaptive Stressor Identification Using Fine-Tuned BER
    [3/19] exclude Architecture-Sensitive Supervised Fine-Tuning for Screen-Con
    ⏳ TPM budget épuisé, attente 1.8s
    ⏳ TPM budget épuisé, attente 7.3s
    [4/19] exclude Cost-Aware Model Selection for Text Classification: Multi-Ob
    ⏳ TPM budget épuisé, attente 1.3s
    ⏳ TPM budget épuisé, attente 7.2s
    [5/19] include Named Entity Recognition with GLiNER and Relation Extraction
    ⏳ TPM budget épuisé, attente 1.2s
    ⏳ TPM budget épuisé, attente 7.6s
    [6/19] include GLiNER-Relex: A Unified Framework for Joint Named Entity Rec
    [7/19] include GLiNER-Relex: A Unified Framework for Joint Named Entity Rec
    ⏳ TPM budget épuisé, attente 0.8s
    ⏳ TPM budget épuisé, attente 7.9s
    [8/19] include Query-Based Named Entity 

RuntimeError: Quota journalier Groq atteint : Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01j74cnq3xfm78jssqp9qzk63h` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199839, Requested 622. Please try again in 3m19.152s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [16]:
import os, importlib, time
from pathlib import Path
from IPython.display import Markdown, display

# Bascule sur deux modèles avec quotas journaliers distincts et valides sur Groq
os.environ["MODEL_A"] = "openai/gpt-oss-20b"
os.environ["MODEL_B"] = "qwen/qwen3.8-27b"

import review_graph
importlib.reload(review_graph)

# IMPORTANT : reset_topic_cache=False pour reprendre directement à l'article 16/19 !
result = run_auto(
    question_ner,
    responses=["ok", "ok"],
    n_per_query=3,
    reset_topic_cache=False,
)

print("Dossier     :", result.get("out_dir"))
print("PRISMA      :", result.get("prisma"))
print("Inclus      :", len(result.get("included", [])))
print("Extraits    :", len(result.get("extraction", [])))
display(Markdown(result.get("report") or ""))


--- INTERRUPT #1 ---
{
  "validate_protocol": {
    "objective": "To compare the performance of different information extraction approaches—fine‑tuned encoders, GLiNER, prompted large language models (LLMs), and LoRA‑fine‑tuned LLMs—on named entity recognition (NER) tasks, focusing on accuracy, computational cost, and out‑of‑domain generalization.",
    "inclusion_criteria": [
      "Empirical studies reporting NER performance metrics (e.g., F1, precision, recall) for at least one of the four approaches listed above",
      "Studies that provide computational cost information (e.g., training time, inference latency, GPU/CPU usage) or sufficient details to estimate it",
      "Studies that evaluate out‑of‑domain generalization using a held‑out dataset or cross‑domain benchmark",
      "Publications indexed in O
→ réponse auto : 'ok'
    OpenAlex 'fine tuned encoder named entity recognition' → 3 hits
    arXiv    'fine tuned encoder named entity recognition' → 3 hits
    OpenAlex 'GLiNE

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

    ✅ Modèle d'embedding chargé
    ✅ Index RAG prêt : 23 documents (Chroma dim=384 + BM25)
    [1/23] include A Fine-Tuned Bidirectional Encoder Representations From Tran
    [2/23] include Treating Decoder-Only LLMs as Encoders: A Simple and Effecti
    ⏳ TPM budget épuisé, attente 5.6s
    [3/23] include BanglaCoNER: Towards Robust Bangla Complex Named Entity Reco
    [4/23] include Query-Based Named Entity Recognition
    ⏳ TPM budget épuisé, attente 2.6s
    ⏳ TPM budget épuisé, attente 5.9s
    [5/23] include A Morpho-Syntactically Informed LSTM-CRF Model for Named Ent
    [6/23] include Named Entity Recognition with GLiNER and Relation Extraction
    [7/23] include GLiNER-Relex: A Unified Framework for Joint Named Entity Rec
    ⏳ TPM budget épuisé, attente 1.0s
    ⏳ TPM budget épuisé, attente 6.3s
    [8/23] include GLiNER: Generalist Model for Named Entity Recognition using 
    [9/23] include GLiNER-Relex: A Unified Framework for Joint Named Entity Rec
    ⏳ TPM budget épuis

# Systematic Literature Review:  
**Comparing Information‑Extraction Approaches for Named Entity Recognition: Accuracy, Cost, and Out‑of‑Domain Generalization**

---

## 1. Executive Summary  

This review synthesizes 16 peer‑reviewed studies that compare four families of NER approaches:  
1. **Fine‑tuned encoders** (e.g., BERT, RoBERTa, DeBERTa)  
2. **GLiNER** – a retrieval‑augmented bidirectional transformer  
3. **Prompted large language models (LLMs)** (generation‑based NER)  
4. **LoRA‑fine‑tuned LLMs** – parameter‑efficient fine‑tuning of large LLMs  

Across the evidence, **fine‑tuned encoders** and **GLiNER** achieve the highest in‑domain F1 scores (≈ 0.78–0.89) on standard benchmarks such as CoNLL‑2003 and OntoNotes. **Prompted LLMs** generally lag in raw accuracy (≈ 0.70–0.75) but offer lower inference cost per token and better zero‑shot adaptability in the limited studies that report these metrics. **LoRA‑fine‑tuned LLMs** provide a middle ground: comparable accuracy to fine‑tuned encoders (≈ 0.72–0.83) while reducing the number of trainable parameters and inference cost, though detailed cost analyses are sparse.  

Out‑of‑domain (OOD) generalization is most robust for **GLiNER** and **LoRA‑fine‑tuned LLMs** in the few studies that evaluate cross‑domain transfer, whereas fine‑tuned encoders show a larger drop in performance when applied to unseen domains.  

---

## 2. Methodology & PRISMA Flow  

| Search Phase | Records | Notes |
|--------------|---------|-------|
| **Databases searched** | OpenAlex, arXiv | 2018–2026, English language |
| **Initial records identified** | 30 |  |
| **Duplicates removed** | 7 |  |
| **Records screened (title/abstract)** | 23 |  |
| **Full‑text assessed** | 23 |  |
| **Records excluded** | 7 |  |
| **Studies included** | 16 |  |

The PRISMA 2020 flow diagram (not shown) reflects the counts above. All included studies were peer‑reviewed and reported in English between 2019 and 2026.

---

## 3. Characteristics & Quality of Included Studies  

| ID | Study Design | Data / Population | Method | Key Results | Limitations | Quality Score (0–5) |
|----|--------------|-------------------|--------|-------------|-------------|---------------------|
| [1] | Model development & internal validation | Food‑related text corpus | Fine‑tuned BERT | Hybrid F1 = 0.87 | Single domain, no OOD testing | 2 |
| [2] | Experimental comparative | CoNLL‑2003, OntoNotes | Fine‑tune decoder‑only LLM | Comparable accuracy to encoders | No GLiNER/LoRA comparison | 3 |
| [3] | Challenge solution | BanglaCoNER dataset | Fine‑tuned encoder (hybrid) | Hybrid F1 = 0.82 | No GLiNER/LLM comparison | 2 |
| [4] | Experimental comparison | English news | Query‑based MRC | Hybrid 0.74, Dense 0.66, BM25 0.87 | No cost/OOD data | 3 |
| [5] | Observational experimental | Bulgarian NER corpus | LSTM‑CRF | POS tags improve accuracy | No cost/OOD data | 3 |
| [6] | Shared task implementation | GutBrain dataset | Fine‑tuned GLiNER | Robust NER | No accuracy metrics reported | 3 |
| [7] | Experimental evaluation | English biomedical & general | Joint GLiNER‑RE | Competitive NER, better OOD | Limited domains, no cost | 3 |
| [8] | Empirical benchmark | CoNLL, OntoNotes | Fine‑tuned GLiNER | Hybrid 0.89, Dense 0.81 | No cost/OOD | 3 |
| [9] | (duplicate of [7]) | – | – | – | – | – |
| [10] | Experimental comparison | NER datasets (unspecified) | Prompted LLM (generation) | Hybrid 0.75, Dense 0.70 | No cost/OOD data | 3 |
| [15] | Comparative experimental | CoNLL‑2003, OntoNotes, biomedical | LoRA‑fine‑tuned Meta‑Llama‑3‑8B‑Instruct | LoRA‑LLM 83 % F1 (hybrid) | Limited benchmarks, incomplete cost | 4 |
| [16] | Experimental comparative | BC2GM Gene Recognition | LoRA on Llama 3.2, Qwen 2.5 | LoRA outperforms baseline | No direct comparison to other families | 2 |
| [17] | Experimental evaluation | Coal‑mine hydraulic support texts | LoRA‑fine‑tuned LLM | Higher accuracy than baseline | Limited dataset, no cost | 3 |
| [18] | Experimental comparative | BERT‑based NER | Contrastive pre‑training + fine‑tuning | Improved OOD accuracy | No cost data | 4 |
| [19] | Benchmark study | French‑Italian wastewater corpus | Fine‑tuned encoders, GLiNER, prompted LLMs, LoRA‑LLMs | Hybrid 0.87, LoRA‑LLM 0.66 | Small corpus, no cost | 3 |
| [22] | Single‑model CNN+GRU | Chinese NER datasets | Convolutional attention network | F1 ≈ 90 % | No GLiNER/LLM comparison | 4 |

*Note: Study [9] is a duplicate of [7] and is not counted separately.*

---

## 4. Thematic & Comparative Synthesis  

### 4.1 In‑Domain Accuracy  

| Approach | Representative F1 Range | Supporting Studies |
|----------|------------------------|--------------------|
| Fine‑tuned encoders | 0.70–0.89 | [1], [3], [4], [5], [8] |
| GLiNER | 0.81–0.89 | [6], [7], [8] |
| Prompted LLMs | 0.70–0.75 | [10] |
| LoRA‑fine‑tuned LLMs | 0.72–0.83 | [15], [16], [17] |

Fine‑tuned encoders and GLiNER consistently achieve the highest in‑domain F1 scores. Prompted LLMs lag behind, while LoRA‑fine‑tuned LLMs fall between the two encoder‑based families.

### 4.2 Cost (Inference & Training)  

Only two studies provide explicit cost information:

| Study | Cost Metric | Findings |
|-------|-------------|----------|
| [15] | Parameter count & GPU‑hour estimate | LoRA‑LLM uses ~ 8 % of full‑size parameters, reducing inference cost per token by ~ 30 % compared to full fine‑tuning |
| [10] | Token‑level inference cost (reported qualitatively) | Prompted LLMs incur lower per‑token cost than fine‑tuned encoders, but no quantitative data |

Thus, **LoRA‑fine‑tuned LLMs** demonstrate a clear cost advantage, whereas **prompted LLMs** are suggested to be cheaper per token, though evidence is limited.

### 4.3 Out‑of‑Domain Generalization  

| Approach | OOD Evidence | Observed Performance |
|----------|--------------|----------------------|
| Fine‑tuned encoders | [18] | Slight drop in accuracy on unseen domains |
| GLiNER | [7], [8] | Better OOD performance than separate encoders |
| Prompted LLMs | None reported | No OOD data |
| LoRA‑fine‑tuned LLMs | [15], [19] | Comparable or better OOD performance relative to fine‑tuned encoders |

GLiNER and LoRA‑fine‑tuned LLMs show the most robust OOD behavior in the limited studies that evaluate cross‑domain transfer.

### 4.4 Unified Retrieval‑Augmented Framework  

GLiNER is explicitly described as a retrieval‑augmented bidirectional transformer that integrates BM25 retrieval with hybrid and dense representations [8]. No other study claims a unified retrieval‑augmented architecture.

### 4.5 Zero‑Shot Adaptability  

Only the GLiNER‑RElex study [7] reports zero‑shot extraction capabilities, showing competitive NER accuracy without domain‑specific fine‑tuning. Prompted LLMs are theoretically zero‑shot but lack empirical evidence in the included literature.

---

## 5. Limitations & Future Directions  

1. **Sparse Cost Data** – Only one study ([15]) provides quantitative cost metrics. Future work should systematically report GPU‑hours, memory usage, and token‑level inference costs across all approaches.  
2. **Limited OOD Evaluation** – OOD generalization is assessed in only three studies ([7], [15],

## References

- **[1]** A Fine-Tuned Bidirectional Encoder Representations From Transformers Model for Food Named-Entity Recognition: Algorithm Development and Validation (2021). *OPENALEX*. https://doi.org/10.2196/28229
- **[2]** Treating Decoder-Only LLMs as Encoders: A Simple and Effective Fine-tuning Approach for Named Entity Recognition (2026). *OPENALEX*. https://doi.org/10.18653/v1/2026.bionlp-1.25
- **[3]** BanglaCoNER: Towards Robust Bangla Complex Named Entity Recognition (2023). *ARXIV*. http://arxiv.org/abs/2303.09306v2
- **[4]** Query-Based Named Entity Recognition (2019). *ARXIV*. http://arxiv.org/abs/1908.09138v2
- **[5]** A Morpho-Syntactically Informed LSTM-CRF Model for Named Entity Recognition (2019). *ARXIV*. http://arxiv.org/abs/1908.10261v1
- **[6]** Named Entity Recognition with GLiNER and Relation Extraction with LLMs (2025). *OPENALEX*. https://openalex.org/W7111820177
- **[7]** GLiNER-Relex: A Unified Framework for Joint Named Entity Recognition and Relation Extraction (2026). *OPENALEX*. https://doi.org/10.48550/arxiv.2605.10108
- **[8]** GLiNER: Generalist Model for Named Entity Recognition using Bidirectional Transformer (2024). *OPENALEX*. https://doi.org/10.18653/v1/2024.naacl-long.300
- **[9]** GLiNER-Relex: A Unified Framework for Joint Named Entity Recognition and Relation Extraction (2026). *ARXIV*. http://arxiv.org/abs/2605.10108v1
- **[10]** GPT-NER: Named Entity Recognition via Large Language Models (2025). *OPENALEX*. https://doi.org/10.18653/v1/2025.findings-naacl.239
- **[15]** LLM-NER: Advancing Named Entity Recognition with LoRA+ Fine-Tuned Large Language Models (2025). *OPENALEX*. https://doi.org/10.1109/iccai66501.2025.00063
- **[16]** Efficient Model Fine-Tuning with LoRA for Biomedical Named Entity Recognition (2025). *OPENALEX*. https://doi.org/10.1109/icaibd64986.2025.11082049
- **[17]** Entity Recognition for Coal Mine Hydraulic Support Installation Process Driven by LLM LoRA Fine-Tuning (2026). *OPENALEX*. https://doi.org/10.3390/app16041943
- **[18]** Contrastive fine-tuning to improve generalization in deep NER (2022). *OPENALEX*. https://doi.org/10.28995/2075-7182-2022-21-70-80
- **[19]** Novel Benchmark for NER in the Wastewater and Stormwater Domain (2025). *OPENALEX*. https://doi.org/10.1109/cist65886.2025.11224095
- **[22]** CAN-NER: Convolutional Attention Network for Chinese Named Entity Recognition (2019). *ARXIV*. http://arxiv.org/abs/1904.02141v3


## 8. Exportation des livrables (Archive ZIP)

In [17]:
import shutil
from pathlib import Path
from google.colab import files

out_dir = Path(result["out_dir"])
bundle_dir = Path("/content/livrable_revue")
bundle_dir.mkdir(exist_ok=True)

shutil.copy("/content/review_graph.py", bundle_dir / "review_graph.py")
if Path("/content/llm_cache.json").exists():
    shutil.copy("/content/llm_cache.json", bundle_dir / "llm_cache.json")
shutil.copytree(out_dir, bundle_dir / out_dir.name, dirs_exist_ok=True)

zip_path = shutil.make_archive("/content/projet_revue_systematique", "zip", bundle_dir)
print("✅ Archive prête :", zip_path)
files.download(zip_path)

✅ Archive prête : /content/projet_revue_systematique.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Évaluation : accord inter-relecteurs (Kappa), latence et RAG hybride

In [18]:
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

out_dir = Path(result["out_dir"])

# A. Accord inter-relecteurs (Relecteur A vs Relecteur B)
screen_data = json.loads((out_dir / "screening.json").read_text(encoding="utf-8"))
votes_a = [s["A"]["decision"] for s in screen_data]
votes_b = [s["B"]["decision"] for s in screen_data]
agreements = sum(1 for a, b in zip(votes_a, votes_b) if a == b)
kappa = cohen_kappa_score(votes_a, votes_b) if len(set(votes_a + votes_b)) > 1 else 1.0

print(f"📊 Corpus screened        : {len(screen_data)} articles")
print(f"🤝 Accord brut A / B      : {agreements}/{len(screen_data)} ({agreements/len(screen_data):.1%})")
print(f"📐 Kappa de Cohen (A/B)   : {kappa:.3f}")
print(f"⚖️ Conflits résolus (OR)  : {sum(s['conflict'] for s in screen_data)} inclus via vote inclusif")

# B. Temps d'exécution par étape depuis trace.jsonl
traces = [json.loads(line) for line in (out_dir / "trace.jsonl").read_text(encoding="utf-8").splitlines()]
if len(traces) >= 2:
    print(f"\n⏱️ Durée totale du graphe : {traces[-1]['t'] - traces[0]['t']:.1f}s")
    for prev, curr in zip(traces, traces[1:]):
        print(f"   - {curr['agent']:16s} ({curr['event']:18s}) : +{curr['t'] - prev['t']:.1f}s")

# C. Démonstration du RAG hybride (Chroma + BM25) sans appel LLM
hits = review_graph.retrieve_top_k("GLiNER zero-shot out-of-domain generalization", result["index"], k=3)
print("\n🔍 Top-3 RAG hybride (Chroma + BM25) :")
for h in hits:
    print(f"   [{h['id']}] hybrid={h['score']:.3f} (dense={h['dense_score']:.3f}, bm25={h['bm25_score']:.3f}) — {h['title'][:65]}")

📊 Corpus screened        : 23 articles
🤝 Accord brut A / B      : 14/23 (60.9%)
📐 Kappa de Cohen (A/B)   : 0.321
⚖️ Conflits résolus (OR)  : 9 inclus via vote inclusif

⏱️ Durée totale du graphe : 341.9s
   - human            (protocol_approved ) : +0.0s
   - search_strategy  (queries           ) : +0.4s
   - retrieval        (query_hits        ) : +1.8s
   - retrieval        (query_hits        ) : +0.8s
   - retrieval        (query_hits        ) : +3.5s
   - retrieval        (query_hits        ) : +1.4s
   - retrieval        (query_hits        ) : +1.7s
   - retrieval        (done              ) : +4.7s
   - retrieval        (index_built       ) : +0.0s
   - screening        (done              ) : +123.7s
   - human            (screening_reviewed) : +0.0s
   - extraction       (done              ) : +63.0s
   - synthesis        (draft_delivered   ) : +59.3s
   - critic           (verdict           ) : +1.2s
   - synthesis        (draft_delivered   ) : +71.3s
   - critic           (ver